# Agent with Tool Retry middleware

### SETUP
Install Langchain dependencies

In [ ]:
%pip install -U langchain langchain-core langchain-community langchain-ollama redis --quiet

In [1]:
import redis
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langgraph.checkpoint.memory import InMemorySaver

We will access a local Redis server running in Docker. Connect to the database and populate the database with some entries

In [2]:
r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.set("user:001", "Name: Joe Bloggs, Role: Engineer")
r.set("user:002", "Name: Jane Ryan, Role: Designer")

True

Instantiate the model

In [3]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

Define 2 tools to read data from Redis

In [4]:
@tool
def get_user_data(user_id: str):
    """Retrieves user data from Redis given a user ID."""
    return r.get(f"user:{user_id}")

Give the agent "memory"

In [5]:
checkpointer = InMemorySaver()

Create the agent

In [15]:
SYSTEM_PROMPT = """You are a helpful assistant that interacts with a Redis database.
You have access to a tool:
- get_user_data: use this to get data about a specific user"""

agent = create_agent(
    model=llm,
    tools=[get_user_data],
    system_prompt=SYSTEM_PROMPT,
        middleware=[
        ToolRetryMiddleware(
            max_retries=2,
            #backoff_factor=2.0,
            initial_delay=1.0,
        ),
    ],
    checkpointer=checkpointer
)

### TESTING
First, let's try it out with Redis up and running

In [7]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the role of user 001?"}]},
    config={"configurable": {"thread_id": "00001"}}
)

In [8]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='What is the role of user 001?', additional_kwargs={}, response_metadata={}, id='b85e956b-ba10-4548-bdf8-586d96706317'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-14T07:15:18.8382128Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10206768700, 'load_duration': 7527039700, 'prompt_eval_count': 117, 'prompt_eval_duration': 142916500, 'eval_count': 178, 'eval_duration': 2165277200, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e2556-d6d4-7510-b07e-0ff82415b190-0', tool_calls=[{'name': 'get_user_data', 'args': {'user_id': '001'}, 'id': '26697be9-1c74-4e08-9131-3a125effbae6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 178, 'total_tokens': 295}),
              ToolMessage(content='Name: Joe Bloggs, Role: Engineer', name='get_user_data', id='f39defc0-fdcf-4256-bbd8-5160

Now we try with Redis stopped

In [16]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the role of user 002?"}]},
    config={"configurable": {"thread_id": "00001"}}
)

In [17]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='What is the role of user 001?', additional_kwargs={}, response_metadata={}, id='b85e956b-ba10-4548-bdf8-586d96706317'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-14T07:15:18.8382128Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10206768700, 'load_duration': 7527039700, 'prompt_eval_count': 117, 'prompt_eval_duration': 142916500, 'eval_count': 178, 'eval_duration': 2165277200, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e2556-d6d4-7510-b07e-0ff82415b190-0', tool_calls=[{'name': 'get_user_data', 'args': {'user_id': '001'}, 'id': '26697be9-1c74-4e08-9131-3a125effbae6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 178, 'total_tokens': 295}),
              ToolMessage(content='Name: Joe Bloggs, Role: Engineer', name='get_user_data', id='f39defc0-fdcf-4256-bbd8-5160

In [11]:
for m in response["messages"][-4:]:
    print(m.pretty_print())

================================ Human Message =================================

What is the role of user 002?
None
================================== Ai Message ==================================
Tool Calls:
  get_user_data (7e205569-712e-4c36-bcb5-9272c3e279ca)
 Call ID: 7e205569-712e-4c36-bcb5-9272c3e279ca
  Args:
    user_id: 002
None
================================= Tool Message =================================
Name: get_user_data

Tool 'get_user_data' failed after 4 attempts with ConnectionError: Error 10061 connecting to localhost:6379. No connection could be made because the target machine actively refused it.. Please try again.
None
================================== Ai Message ==================================

I'm sorry, but I was unable to retrieve the role for user 002. The connection to the database failed. Could you please try asking again later?
None


In [12]:
agent2 = create_agent(
    model=llm,
    tools=[get_user_data],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

This is the behavior for an agent without the tool retry middleware

In [13]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the role of user 001?"}]},
    config={"configurable": {"thread_id": "00002"}}
)

In [14]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='What is the role of user 001?', additional_kwargs={}, response_metadata={}, id='4521562f-2667-4ecd-9e2f-b9aae4998069'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-14T07:22:50.8329132Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3119707900, 'load_duration': 277760000, 'prompt_eval_count': 117, 'prompt_eval_duration': 190705800, 'eval_count': 211, 'eval_duration': 2530149900, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e255d-d81f-7f51-893a-e373b07620fe-0', tool_calls=[{'name': 'get_user_data', 'args': {'user_id': '001'}, 'id': '45f8d360-3093-4236-902f-987d71c7b5dd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 211, 'total_tokens': 328}),
              ToolMessage(content="Tool 'get_user_data' failed after 4 attempts with ConnectionError: Error 10061 connecting t